# 01 - Análisis Exploratorio de Datos (EDA) y Partición Estratificada
**Proyecto:** Sentix-Hotel
**Responsable:** José (Capa 1: Datos y EDA)

Este notebook realiza el análisis exploratorio del dataset de reseñas de hoteles en español,
verificando el balance de clases, longitudes de texto, valores faltantes y efectuando
la partición estratificada 80/20 train/test fijando semilla aleatoria para garantizar reproducibilidad.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Configuración de entorno y rutas
PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))
from src.config import RAW_DATASET_PATH, DATA_PROCESSED_DIR, RANDOM_STATE, TEST_SIZE

sns.set_theme(style='whitegrid')
print(f'Ruta del dataset crudo: {RAW_DATASET_PATH}')

## 1. Carga del Dataset y Estructura Básica

In [ ]:
df_reviews = pd.read_csv(RAW_DATASET_PATH)
print('Filas y Columnas:', df_reviews.shape)
print('Valores nulos:\n', df_reviews.isnull().sum())
df_reviews.head()

## 2. Distribución y Balance de Clases de Sentimiento

In [ ]:
class_counts = df_reviews['sentiment_label'].value_counts()
print(class_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=['#2ecc71', '#e74c3c'])
plt.title('Distribución de Clases de Sentimiento')
plt.xlabel('Sentimiento')
plt.ylabel('Cantidad de Reseñas')
plt.tight_layout()
plt.show()

## 3. Análisis de Longitud de Textos (Palabras y Caracteres)

In [ ]:
df_reviews['char_count'] = df_reviews['review_text'].astype(str).str.len()
df_reviews['word_count'] = df_reviews['review_text'].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df_reviews, x='word_count', hue='sentiment_label', kde=True, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Palabras por Reseña')
axes[0].set_xlim(0, 300)

sns.boxplot(data=df_reviews, x='sentiment_label', y='word_count', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Palabras por Clase de Sentimiento')
axes[1].set_ylim(0, 300)
plt.tight_layout()
plt.show()

print(df_reviews.groupby('sentiment_label')['word_count'].describe())

## 4. División Estratificada Train / Test (80/20)
Se fija la semilla aleatoria en 42 y se estratifica por sentimiento para garantizar particiones reproducibles.

In [ ]:
train_df, test_df = train_test_split(
    df_reviews,
    test_size=TEST_SIZE,
    stratify=df_reviews['sentiment'],
    random_state=RANDOM_STATE
)

print('Train size:', len(train_df), 'Test size:', len(test_df))
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(DATA_PROCESSED_DIR / 'train_reviews.csv', index=False, encoding='utf-8')
test_df.to_csv(DATA_PROCESSED_DIR / 'test_reviews.csv', index=False, encoding='utf-8')
print('Particiones guardadas en data/processed/')